In [ ]:
import sys
!{sys.executable} -m pip install pyserial numpy matplotlib ipympl

In [ ]:
import serial
import struct
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# 1. 환경 설정 (본인의 COM 포트 번호로 수정하세요)
PORT = 'COM18' 
BAUD = 115200
PACKET_SIZE = 18
HEADER = 0xFE
FOOTER = 0xFF

# 2. 그래프 설정 (%matplotlib widget 필수!)
%matplotlib widget
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7))
plt.subplots_adjust(hspace=0.4)

# 데이터 보관함 (최근 200개 데이터 표시)
maxlen = 200
err_x, err_y = [0]*maxlen, [0]*maxlen
out_x, out_y = [0]*maxlen, [0]*maxlen

# 라인 객체 생성
line_err_x, = ax1.plot(err_x, label='Error X', color='red', linewidth=1)
line_err_y, = ax1.plot(err_y, label='Error Y', color='blue', linewidth=1)
line_out_x, = ax2.plot(out_x, label='Output X', color='orange', linewidth=1)
line_out_y, = ax2.plot(out_y, label='Output Y', color='green', linewidth=1)

# 그래프 꾸미기
ax1.set_title("Tracking Error (Pixels)"); ax1.set_ylim(-350, 350); ax1.grid(True); ax1.legend(loc='upper right')
ax2.set_title("PID Output (Delta Degree)"); ax2.set_ylim(-15, 15); ax2.grid(True); ax2.legend(loc='upper right')

# 3. 시리얼 포트 열기
try:
    if 'ser' in locals() and ser.is_open: ser.close() # 기존 연결 있으면 닫기
    ser = serial.Serial(PORT, BAUD, timeout=0.01)
    print(f"Connected to {PORT}!")
except Exception as e:
    print(f"연결 실패: {e}. 포트 번호를 확인하세요.")

# 4. 업데이트 함수 (실시간 데이터 파싱)
def update(frame):
    global err_x, err_y, out_x, out_y
    
    # 패킷 사이즈만큼 데이터가 들어왔는지 확인
    while ser.in_waiting >= PACKET_SIZE:
        if ord(ser.read(1)) == HEADER: # 헤더 확인
            payload = ser.read(PACKET_SIZE - 1)
            if payload[-1] == FOOTER: # 푸터 확인
                # 데이터 추출 (f: float 4개)
                vals = struct.unpack('<ffff', payload[:-1])
                ex, ox, ey, oy = vals
                
                # 데이터 갱신
                err_x.pop(0); err_x.append(ex)
                err_y.pop(0); err_y.append(ey)
                out_x.pop(0); out_x.append(ox)
                out_y.pop(0); out_y.append(oy)
                
                # 그래프 선 업데이트
                line_err_x.set_ydata(err_x)
                line_err_y.set_ydata(err_y)
                line_out_x.set_ydata(out_x)
                line_out_y.set_ydata(out_y)
                
    return line_err_x, line_err_y, line_out_x, line_out_y

# 5. 애니메이션 시작
ani = FuncAnimation(fig, update, interval=20, blit=True, cache_frame_data=False)
plt.show()

In [ ]:
import serial
import struct
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# 1. 설정
PORT = 'COM18' 
BAUD = 115200
PACKET_SIZE = 18

# 2. 인터랙티브 모드 활성화 및 피규어 생성
plt.close('all') # 열려있는 그래프 모두 닫기
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6))

# 데이터 버퍼
maxlen = 200
err_x, err_y = [0.0]*maxlen, [0.0]*maxlen
out_x, out_y = [0.0]*maxlen, [0.0]*maxlen

line1, = ax1.plot(err_x, 'r-', label='Err X')
line1_y, = ax1.plot(err_y, 'b-', label='Err Y')
line2, = ax2.plot(out_x, 'r--', label='Out X')
line2_y, = ax2.plot(out_y, 'b--', label='Out Y')

ax1.set_ylim(-350, 350); ax1.legend(); ax1.grid(True)
ax2.set_ylim(-15, 15); ax2.legend(); ax2.grid(True)

# 시리얼 연결
try:
    if 'ser' in locals():
        ser.close() # 이미 열려있다면 닫기
    ser = serial.Serial(PORT, BAUD, timeout=0.01)
    print(f"Successfully connected to {PORT}")
except Exception as e:
    print(f"Error: {e}")
    print("TIP: 다른 프로그램이 포트를 쓰고 있는지 확인하거나 주피터 커널을 Restart 하세요.")

def update(frame):
    while ser.in_waiting >= PACKET_SIZE:
        header = ser.read(1)
        if header == b'\xfe':
            data = ser.read(PACKET_SIZE - 1)
            if data[-1] == 0xff:
                ex, ox, ey, oy = struct.unpack('<ffff', data[:-1])
                
                print(f"RX: {ex:.2f}, {ey:.2f}")

                err_x.pop(0); err_x.append(ex)
                err_y.pop(0); err_y.append(ey)
                out_x.pop(0); out_x.append(ox)
                out_y.pop(0); out_y.append(oy)
                
                line1.set_ydata(err_x)
                line1_y.set_ydata(err_y)
                line2.set_ydata(out_x)
                line2_y.set_ydata(out_y)
                
    return line1, line1_y, line2, line2_y

# [중요] ani 변수를 전역변수처럼 유지해야 합니다.
ani = FuncAnimation(fig, update, interval=30, blit=True, cache_frame_data=False)
plt.show()

# 첫 성공 로직 - 실시간 그래프 보임 위 - error, 아래 - pid output

In [ ]:
import serial
import struct
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# 1. 설정
PORT = 'COM18' 
BAUD = 115200
PACKET_SIZE = 18

# 2. 인터랙티브 모드 및 그래프 초기화
%matplotlib qt 
plt.close('all')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6))

maxlen = 200
err_x, err_y = [0.0]*maxlen, [0.0]*maxlen
out_x, out_y = [0.0]*maxlen, [0.0]*maxlen

line1, = ax1.plot(err_x, 'r-', label='Err X')
line1_y, = ax1.plot(err_y, 'b-', label='Err Y')
line2, = ax2.plot(out_x, 'r--', label='Out X')
line2_y, = ax2.plot(out_y, 'b--', label='Out Y')

ax1.set_ylim(-350, 350); ax1.legend(); ax1.grid(True)
ax2.set_ylim(-15, 15); ax2.legend(); ax2.grid(True)

# 3. 시리얼 연결 (재시도 로직 포함)
if 'ser' in locals() and ser.is_open:
    ser.close()
ser = serial.Serial(PORT, BAUD, timeout=0.001) # 타임아웃을 아주 짧게 설정
ser.reset_input_buffer() # 시작 전 버퍼 비우기

def update(frame):
    # [핵심] 데이터가 너무 많이 쌓여있으면 루프가 느려짐
    # 버퍼에 데이터가 너무 많으면 최근 데이터만 남기고 비움
    if ser.in_waiting > PACKET_SIZE * 10:
        ser.reset_input_buffer()
        return line1, line1_y, line2, line2_y

    while ser.in_waiting >= PACKET_SIZE:
        header = ser.read(1)
        if header == b'\xfe':
            data = ser.read(PACKET_SIZE - 1)
            if data[-1] == 0xff:
                ex, ox, ey, oy = struct.unpack('<ffff', data[:-1])
                
                # 데이터 갱신
                err_x.pop(0); err_x.append(ex)
                err_y.pop(0); err_y.append(ey)
                out_x.pop(0); out_x.append(ox)
                out_y.pop(0); out_y.append(oy)
                
                line1.set_ydata(err_x)
                line1_y.set_ydata(err_y)
                line2.set_ydata(out_x)
                line2_y.set_ydata(out_y)
                
    return line1, line1_y, line2, line2_y

# blit=False로 설정하면 약간 더 무겁지만 동기화 이슈가 줄어듭니다.
ani = FuncAnimation(fig, update, interval=20, blit=False, cache_frame_data=False)
plt.show()

# kp, ki, kd 표시

In [5]:
import serial
import struct
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import time

# ==========================================
# 1. Recording Parameters (Update these to match your STM32 code)
# ==========================================
KP, KI, KD = 0.005, 0.0, 0.0  # Update these before recording!
PORT = 'COM18' 
BAUD = 115200
PACKET_SIZE = 18
DT = 0.020  # 20ms sampling period

# ==========================================
# 2. Plot Initialization
# ==========================================
%matplotlib qt 
plt.close('all')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

# Data Buffers
maxlen = 200
err_x = [0.0]*maxlen
out_x = [0.0]*maxlen

line1, = ax1.plot(err_x, 'r-', label='Tracking Error (X)')
line2, = ax2.plot(out_x, 'b--', label='PID Output (X)')

ax1.set_ylim(-350, 350); ax1.grid(True); ax1.legend(loc='upper right')
ax2.set_ylim(-20, 20); ax2.grid(True); ax2.legend(loc='upper right')
ax1.set_ylabel("Error (px)")
ax2.set_ylabel("Output (deg)")

# Period Calculation Variables
peak_times = []
current_period = 0.0

# ==========================================
# 3. Serial Connection
# ==========================================
try:
    if 'ser' in locals() and ser.is_open: ser.close()
    ser = serial.Serial(PORT, BAUD, timeout=0.001)
    ser.reset_input_buffer()
    print(f"Connected to {PORT}")
except Exception as e:
    print(f"Connection Error: {e}")

# ==========================================
# 4. Period Detection Logic (Peak-to-Peak)
# ==========================================
def calculate_period(data_buffer):
    """Detects oscillation period using peak-to-peak timing"""
    global peak_times, current_period
    
    # Simple peak detection: check if the last point is a local maximum
    if len(data_buffer) > 3:
        if data_buffer[-2] > data_buffer[-3] and data_buffer[-2] > data_buffer[-1] and data_buffer[-2] > 50:
            now = time.time()
            peak_times.append(now)
            if len(peak_times) > 2:
                # Calculate average period of last 3 peaks
                current_period = (peak_times[-1] - peak_times[-2])
                peak_times = peak_times[-5:] # Keep last 5 peaks
    return current_period

# ==========================================
# 5. Main Update Loop
# ==========================================
def update(frame):
    global err_x, out_x, current_period

    # Flush buffer if lagging
    if ser.in_waiting > PACKET_SIZE * 10:
        ser.reset_input_buffer()

    while ser.in_waiting >= PACKET_SIZE:
        header = ser.read(1)
        if header == b'\xfe':
            data = ser.read(PACKET_SIZE - 1)
            if data[-1] == 0xff:
                # Unpack: ErrorX, OutX, ErrorY, OutY
                ex, ox, ey, oy = struct.unpack('<ffff', data[:-1])
                
                err_x.pop(0); err_x.append(ex)
                out_x.pop(0); out_x.append(ox)
                
                # Update Period
                period = calculate_period(err_x)
                
                # Update Lines
                line1.set_ydata(err_x)
                line2.set_ydata(out_x)
                
                # Update Title with Parameters and Results
                title_str = (f"PID Tuning | Kp:{KP:.4f}, Ki:{KI:.4f}, Kd:{KD:.4f}\n"
                             f"Oscillation Period: {period:.3f}s | Freq: {1/period if period>0 else 0:.1f}Hz")
                fig.suptitle(title_str, fontsize=12, fontweight='bold', color='darkred')
                
    return line1, line2

ani = FuncAnimation(fig, update, interval=20, blit=False, cache_frame_data=False)
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust for title space
plt.show()

Connected to COM18


# 글자 크기 확대

In [7]:
import serial
import struct
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import time

# ==========================================================
# 1. RECORDING PARAMETERS (Update these before each run!)
# ==========================================================
KP = 0.0050
KI = 0.0000
KD = 0.0000
PORT = 'COM18' 
BAUD = 115200
PACKET_SIZE = 18
DT = 0.020  # 20ms Control Loop Period

# ==========================================================
# 2. PLOT INITIALIZATION
# ==========================================================
%matplotlib qt 
plt.close('all')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 9))

# Data Buffers
maxlen = 200
err_x = [0.0] * maxlen
out_x = [0.0] * maxlen

# Line Objects
line1, = ax1.plot(err_x, 'r-', linewidth=1.5, label='Tracking Error (X)')
line2, = ax2.plot(out_x, 'b--', linewidth=1.5, label='PID Output (X)')

# Graph Styling
ax1.set_ylim(-350, 350)
ax1.grid(True, which='both', linestyle='--', alpha=0.5)
ax1.legend(loc='upper right')
ax1.set_ylabel("Error (px)", fontsize=12)

ax2.set_ylim(-20, 20)
ax2.grid(True, which='both', linestyle='--', alpha=0.5)
ax2.legend(loc='upper right')
ax2.set_ylabel("PID Output (deg)", fontsize=12)

# Global Variables for Period Calculation
peak_times = []
current_period = 0.0

# ==========================================================
# 3. SERIAL CONNECTION
# ==========================================================
try:
    if 'ser' in locals() and ser.is_open:
        ser.close()
    ser = serial.Serial(PORT, BAUD, timeout=0.001)
    ser.reset_input_buffer()
    print(f"Successfully connected to {PORT}")
except Exception as e:
    print(f"Connection Error: {e}")

# ==========================================================
# 4. CALCULATION LOGIC
# ==========================================================
def calculate_period(data_buffer):
    """Detects oscillation period using peak-to-peak timing"""
    global peak_times, current_period
    
    # Simple Peak Detection: Look for local maximum > 50px
    if len(data_buffer) > 3:
        if data_buffer[-2] > data_buffer[-3] and data_buffer[-2] > data_buffer[-1] and data_buffer[-2] > 50:
            now = time.time()
            # Prevent double detection of the same peak
            if not peak_times or (now - peak_times[-1] > 0.1):
                peak_times.append(now)
                if len(peak_times) >= 2:
                    current_period = peak_times[-1] - peak_times[-2]
                peak_times = peak_times[-5:] # Keep last 5 peaks
    return current_period

# ==========================================================
# 5. MAIN UPDATE LOOP (Animation)
# ==========================================================
def update(frame):
    global err_x, out_x, current_period

    # Prevent lag by clearing old buffer
    if ser.in_waiting > PACKET_SIZE * 15:
        ser.reset_input_buffer()

    while ser.in_waiting >= PACKET_SIZE:
        header = ser.read(1)
        if header == b'\xfe':
            data = ser.read(PACKET_SIZE - 1)
            if data[-1] == 0xff:
                # Unpack: ErrorX, OutX, ErrorY, OutY
                ex, ox, ey, oy = struct.unpack('<ffff', data[:-1])
                
                err_x.pop(0); err_x.append(ex)
                out_x.pop(0); out_x.append(ox)
                
                # Update Calculations
                period = calculate_period(err_x)
                
                # Update Graph Lines
                line1.set_ydata(err_x)
                line2.set_ydata(out_x)
                
                # Update Recording-Optimized Title
                freq = 1.0 / period if period > 0 else 0
                title_str = (f"PID TUNING | Kp: {KP:.4f}  Ki: {KI:.4f}  Kd: {KD:.4f}\n"
                             f"PERIOD: {period:.3f}s  |  FREQ: {freq:.1f}Hz")
                
                fig.suptitle(title_str, 
                             fontsize=20,          # Extra large font
                             fontweight='bold', 
                             color='yellow',       # High visibility
                             backgroundcolor='black') # Solid background for video
                
    return line1, line2

# Adjust layout to prevent overlap with the large title
plt.tight_layout(rect=[0, 0.03, 1, 0.88])

# Start Animation
ani = FuncAnimation(fig, update, interval=20, blit=False, cache_frame_data=False)
plt.show()

Successfully connected to COM18


# 파형 검출 변경 - 제로 크로싱

In [12]:
import serial
import struct
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import time

# ==========================================================
# 1. RECORDING PARAMETERS (Update these before each run!)
# ==========================================================
KP = 0.0100
KD = 0.0050
PORT = 'COM18' 
BAUD = 115200
PACKET_SIZE = 18
DT = 0.020  # 20ms Control Loop Period

# ==========================================================
# 2. PLOT INITIALIZATION
# ==========================================================
%matplotlib qt 
plt.close('all')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 9))

# Data Buffers
maxlen = 200
err_x = [0.0] * maxlen
out_x = [0.0] * maxlen

# Line Objects
line1, = ax1.plot(err_x, 'r-', linewidth=1.5, label='Tracking Error (X)')
line2, = ax2.plot(out_x, 'b--', linewidth=1.5, label='PID Output (X)')

# Graph Styling
ax1.set_ylim(-350, 350)
ax1.grid(True, which='both', linestyle='--', alpha=0.5)
ax1.legend(loc='upper right')
ax1.set_ylabel("Error (px)", fontsize=12)

ax2.set_ylim(-20, 20)
ax2.grid(True, which='both', linestyle='--', alpha=0.5)
ax2.legend(loc='upper right')
ax2.set_ylabel("PID Output (deg)", fontsize=12)

# Global Variables for Period Calculation
cross_times = []  # Time stamps when error crosses zero
current_period = 0.0

# ==========================================================
# 3. SERIAL CONNECTION
# ==========================================================
try:
    if 'ser' in locals() and ser.is_open:
        ser.close()
    ser = serial.Serial(PORT, BAUD, timeout=0.001)
    ser.reset_input_buffer()
    print(f"Successfully connected to {PORT}")
except Exception as e:
    print(f"Connection Error: {e}")

# ==========================================================
# 4. ENHANCED CALCULATION LOGIC (Zero-Crossing)
# ==========================================================
def update_period_logic(data_buffer):
    """Detects oscillation period using zero-crossing points"""
    global cross_times, current_period
    
    if len(data_buffer) < 5: return current_period

    # Detect Zero Crossing: Sign of current vs previous value
    # We check if (prev > 0 and curr <= 0) OR (prev < 0 and curr >= 0)
    prev_val = data_buffer[-2]
    curr_val = data_buffer[-1]
    
    # Only detect if there's a significant swing to avoid noise near zero
    if (prev_val * curr_val <= 0) and (abs(prev_val - curr_val) > 2):
        now = time.time()
        
        # Debounce: Ensure crossings are at least 50ms apart
        if not cross_times or (now - cross_times[-1] > 0.05):
            cross_times.append(now)
            
            # 3 points (2 intervals) = 1 Full Cycle (Oscillation)
            if len(cross_times) >= 3:
                # Time between 1st and 3rd crossing is one full wave
                new_period = cross_times[-1] - cross_times[-3]
                
                # Simple low-pass filter for period display stability
                if current_period == 0:
                    current_period = new_period
                else:
                    current_period = current_period * 0.7 + new_period * 0.3
            
            cross_times = cross_times[-10:] # Keep last 10 points
            
    return current_period

# ==========================================================
# 5. MAIN UPDATE LOOP (Animation)
# ==========================================================
def update(frame):
    global err_x, out_x, current_period

    if ser.in_waiting > PACKET_SIZE * 15:
        ser.reset_input_buffer()

    while ser.in_waiting >= PACKET_SIZE:
        header = ser.read(1)
        if header == b'\xfe':
            data = ser.read(PACKET_SIZE - 1)
            if data[-1] == 0xff:
                ex, ox, ey, oy = struct.unpack('<ffff', data[:-1])
                
                err_x.pop(0); err_x.append(ex)
                out_x.pop(0); out_x.append(ox)
                
                # Update Calculations
                period = update_period_logic(err_x)
                
                # Update Graph Lines
                line1.set_ydata(err_x)
                line2.set_ydata(out_x)
                
                # Update Title
                freq = 1.0 / period if period > 0.1 else 0
                title_str = (f"PID TUNING | Kp: {KP:.4f}  Kd: {KD:.4f}\n"
                             f"PERIOD: {period:.3f}s  |  FREQ: {freq:.1f}Hz")
                
                fig.suptitle(title_str, 
                             fontsize=22,          # Very large for video
                             fontweight='bold', 
                             color='yellow',       
                             backgroundcolor='black') 
                
    return line1, line2

# Adjust layout to prevent overlap
plt.tight_layout(rect=[0, 0.03, 1, 0.85])

# Start Animation
ani = FuncAnimation(fig, update, interval=20, blit=False, cache_frame_data=False)
plt.show()

Successfully connected to COM18
